# Analytica Agent Evaluation Metrics

This notebook evaluates Analytica on a fixed set of analytical tasks. It checks whether the system understands the question, chooses the right fields, computes evidence, and returns an answer based on this evidence.



## 1. Purpose

This notebook measures the analytical quality of the current Analytica prototype.

The evaluation uses fixed datasets and fixed questions. Reference values are computed before the system output is checked. They are not copied from the generated answer.

The notebook is used as supporting evidence for the final report and defense materials.


## 2. Evaluation Setup

The evaluation runs in deterministic local mode. A live LLM provider is not called in this run. The runner returns an unavailable status, so InvestigationService uses the product fallback path.

This still tests real product behavior, because the service path is executed. It does not test provider-specific wording, model variation, or live LLM latency.

The numeric tolerance is 1e-6. The retry limit is 1.


In [17]:
from pathlib import Path
import os
import re
import sys
import time
import warnings

project_root = Path.cwd().resolve()
while not (project_root / "source" / "product" / "service.py").exists():
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

warnings.filterwarnings("ignore", message="Core Pydantic V1 functionality.*")
warnings.filterwarnings("ignore", message="Could not infer format.*")
warnings.filterwarnings("ignore", message="The default value of `allowed_objects`.*")

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import HTML, display

from source.product.service import InvestigationService
from source.product.investigation import ArtifactType, InvestigationStatus
from source.product.dataset_registry import DatasetResolutionResult, DatasetScope, InvestigationDatasetEntry, cross_dataset_output

pio.templates.default = "plotly_white"
pd.set_option("display.max_colwidth", 120)

tolerance = 1e-6
retry_limit = 1

colors = {
    "blue": "#1D4E89",
    "dark_blue": "#0B2545",
    "mid_blue": "#4F86C6",
    "light_blue": "#A9C6E8",
    "pale_blue": "#EAF3FB",
    "red": "#B23A48",
    "gray": "#64748B",
    "dark": "#0F172A",
    "caption": "#7DD3FC",
}

In [18]:
def style_table(frame, caption=None, formatters=None):
    styled = frame.style.set_table_styles([
        {"selector": "table", "props": [("border-collapse", "collapse"), ("background", "#FFFFFF"), ("color", "#0F172A"), ("font-family", "Arial, sans-serif")]},
        {"selector": "caption", "props": [("caption-side", "top"), ("font-weight", "800"), ("font-size", "15px"), ("color", "#7DD3FC"), ("text-align", "left"), ("padding", "8px 0")]},
        {"selector": "th", "props": [("background", "#0B2545"), ("color", "#FFFFFF"), ("font-weight", "700"), ("border", "1px solid #CBD5E1"), ("padding", "8px")]},
        {"selector": "td", "props": [("border", "1px solid #E2E8F0"), ("padding", "7px"), ("font-size", "13px"), ("color", "#0F172A"), ("background", "#FFFFFF")]},
        {"selector": "tbody tr:nth-child(even) td", "props": [("background", "#F8FAFC")]},
    ])
    if caption:
        styled = styled.set_caption(caption)
    if formatters:
        styled = styled.format(formatters)
    return styled


def metric_cards(metrics):
    blocks = []
    for item in metrics:
        blocks.append("".join([
            f"<div style='flex:1; min-width:210px; border:1px solid #CBD5E1; border-top:5px solid {item['color']}; border-radius:8px; padding:14px 16px; background:#FFFFFF; box-shadow:0 1px 3px rgba(15,23,42,0.08);'>",
            f"<div style='font-size:12px; color:#64748B; text-transform:uppercase; letter-spacing:.04em; font-weight:700;'>{item['metric']}</div>",
            f"<div style='font-size:28px; color:#0B2545; font-weight:800; margin-top:6px;'>{item['display_value']}</div>",
            f"<div style='font-size:12px; color:#475569; margin-top:5px; line-height:1.35;'>{item['note']}</div>",
            "</div>",
        ]))
    return HTML(f"<div style='display:flex; gap:12px; flex-wrap:wrap; margin:10px 0 16px 0;'>{''.join(blocks)}</div>")


def plotly_layout(fig, title, x_title=None, y_title=None, height=430):
    fig.update_layout(
        title={"text": title, "x": 0.02, "xanchor": "left", "font": {"size": 18, "color": colors["dark"]}},
        paper_bgcolor="#FFFFFF",
        plot_bgcolor="#FFFFFF",
        font={"family": "Arial, sans-serif", "size": 13, "color": colors["dark"]},
        height=height,
        margin={"l": 70, "r": 40, "t": 70, "b": 60},
        legend={"orientation": "h", "yanchor": "bottom", "y": 1.02, "xanchor": "right", "x": 1},
    )
    fig.update_xaxes(title=x_title, showgrid=True, gridcolor="#E2E8F0", zeroline=False, linecolor="#CBD5E1")
    fig.update_yaxes(title=y_title, showgrid=True, gridcolor="#E2E8F0", zeroline=False, linecolor="#CBD5E1")
    return fig


def score_text(value):
    if pd.isna(value):
        return "Not applicable"
    return f"{value:.1%}"

## 3. Test Datasets

The notebook uses two fixed datasets.

The retail dataset is used for sales, profit, discount, shipping cost, grouping, trend, and business KPI tasks.

The health dataset is used for prevalence, risk gap, age, BMI, and indicator tasks.

Both datasets are small enough to inspect by hand. At the same time, they are varied enough to show real mistakes in reasoning and grouping.


In [19]:
retail_df = pd.DataFrame({
    "Order Date": pd.to_datetime(["2026-01-01", "2026-01-03", "2026-02-01", "2026-02-10", "2026-03-01", "2026-03-12", "2026-04-01", "2026-04-15", "2026-05-01", "2026-05-18", "2026-06-01", "2026-06-20"]),
    "City": ["London", "London", "Paris", "Paris", "Tokyo", "Tokyo", "Berlin", "Berlin", "Madrid", "Madrid", "London", "Tokyo"],
    "Region": ["EMEA", "EMEA", "EMEA", "EMEA", "APAC", "APAC", "EMEA", "EMEA", "EMEA", "EMEA", "EMEA", "APAC"],
    "Category": ["Furniture", "Technology", "Furniture", "Office Supplies", "Technology", "Office Supplies", "Furniture", "Technology", "Office Supplies", "Furniture", "Technology", "Furniture"],
    "Segment": ["Consumer", "Corporate", "Consumer", "Home Office", "Corporate", "Consumer", "Corporate", "Consumer", "Home Office", "Corporate", "Consumer", "Home Office"],
    "Sales": [1000, 2400, 800, 600, 5000, 700, 1200, 2600, 900, 1100, 3000, 1600],
    "Profit": [50, 420, -80, 60, 900, 40, 20, 260, -30, 55, 450, -100],
    "Discount": [0.10, 0.05, 0.25, 0.10, 0.02, 0.15, 0.30, 0.08, 0.35, 0.20, 0.06, 0.40],
    "Shipping Cost": [120, 150, 200, 80, 300, 90, 280, 140, 260, 190, 180, 350],
    "Quantity": [2, 5, 1, 3, 8, 2, 3, 6, 2, 4, 7, 2],
})

health_df = pd.DataFrame({
    "Patient ID": [f"P{i:03d}" for i in range(1, 17)],
    "Age": [24, 29, 34, 41, 48, 52, 61, 67, 23, 37, 44, 56, 63, 70, 31, 46],
    "Age Group": ["18-29", "18-29", "30-44", "30-44", "45-59", "45-59", "60+", "60+", "18-29", "30-44", "45-59", "45-59", "60+", "60+", "30-44", "45-59"],
    "Gender": ["F", "M", "F", "M", "F", "M", "F", "M", "F", "M", "F", "M", "F", "M", "F", "M"],
    "BMI": [22, 28, 26, 31, 33, 29, 35, 38, 21, 30, 32, 27, 36, 41, 24, 34],
    "Smoker": ["No", "Yes", "No", "Yes", "Yes", "No", "No", "Yes", "No", "Yes", "Yes", "No", "No", "Yes", "No", "Yes"],
    "Heart Disease": [0, 0, 0, 1, 1, 0, 1, 1, 0, 1, 1, 0, 1, 1, 0, 1],
    "Blood Pressure": [112, 124, 118, 138, 142, 130, 148, 155, 110, 136, 140, 128, 150, 160, 116, 145],
})

dataset_overview = pd.DataFrame([
    {"dataset": "retail", "rows": len(retail_df), "columns": retail_df.shape[1], "purpose": "business metrics, KPIs, trends, visualizations"},
    {"dataset": "health", "rows": len(health_df), "columns": health_df.shape[1], "purpose": "prevalence, risk gap, age and BMI indicators"},
])

retail_display = retail_df.copy()
retail_display["Order Date"] = retail_display["Order Date"].dt.strftime("%Y-%m-%d")

display(style_table(dataset_overview, "Dataset overview"))
display(style_table(retail_display.head(8), "Retail dataset sample", {"Sales": "{:.2f}", "Profit": "{:.2f}", "Discount": "{:.2f}", "Shipping Cost": "{:.2f}"}))
display(style_table(health_df.head(8), "Health dataset sample", {"BMI": "{:.1f}", "Blood Pressure": "{:.0f}"}))

,dataset,rows,columns,purpose
0,retail,12,10,"business metrics, KPIs, trends, visualizations"
1,health,16,8,"prevalence, risk gap, age and BMI indicators"


,Order Date,City,Region,Category,Segment,Sales,Profit,Discount,Shipping Cost,Quantity
0,2026-01-01,London,EMEA,Furniture,Consumer,1000.00,50.00,0.10,120.00,2
1,2026-01-03,London,EMEA,Technology,Corporate,2400.00,420.00,0.05,150.00,5
2,2026-02-01,Paris,EMEA,Furniture,Consumer,800.00,-80.00,0.25,200.00,1
3,2026-02-10,Paris,EMEA,Office Supplies,Home Office,600.00,60.00,0.10,80.00,3
4,2026-03-01,Tokyo,APAC,Technology,Corporate,5000.00,900.00,0.02,300.00,8
5,2026-03-12,Tokyo,APAC,Office Supplies,Consumer,700.00,40.00,0.15,90.00,2
6,2026-04-01,Berlin,EMEA,Furniture,Corporate,1200.00,20.00,0.30,280.00,3
7,2026-04-15,Berlin,EMEA,Technology,Consumer,2600.00,260.00,0.08,140.00,6


,Patient ID,Age,Age Group,Gender,BMI,Smoker,Heart Disease,Blood Pressure
0,P001,24,18-29,F,22.0,No,0,112
1,P002,29,18-29,M,28.0,Yes,0,124
2,P003,34,30-44,F,26.0,No,0,118
3,P004,41,30-44,M,31.0,Yes,1,138
4,P005,48,45-59,F,33.0,Yes,1,142
5,P006,52,45-59,M,29.0,No,0,130
6,P007,61,60+,F,35.0,No,1,148
7,P008,67,60+,M,38.0,Yes,1,155


## 4. Test Scenario Design

The test set includes both common tasks and harder edge cases.

It covers grouped aggregation, trends, prevalence, outliers, business KPIs, relationships, visualizations, follow-up correction, multi-dataset comparison, and incompatible requests.

The cases are fixed before the run. They are not removed or changed when the system fails.


In [20]:
test_cases = [
    {"case_id": "retail_city_total_sales", "category": "basic grouped aggregation", "dataset": "retail", "question": "Top cities by total Sales.", "expected_intent": "grouped aggregation", "expected_metric": ["Sales"], "expected_grouping": ["City"], "expected_aggregation": "sum", "requires_chart": False, "requires_multi_dataset": False},
    {"case_id": "retail_city_average_sales", "category": "basic grouped aggregation", "dataset": "retail", "question": "Top cities by average Sales.", "expected_intent": "grouped aggregation", "expected_metric": ["Sales"], "expected_grouping": ["City"], "expected_aggregation": "mean", "requires_chart": False, "requires_multi_dataset": False},
    {"case_id": "retail_sales_trend", "category": "trend analysis", "dataset": "retail", "question": "Create a line chart of total Sales over Order Date.", "expected_intent": "trend analysis", "expected_metric": ["Sales"], "expected_grouping": ["Order Date"], "expected_aggregation": "sum", "requires_chart": True, "expected_chart_type": "line", "requires_multi_dataset": False},
    {"case_id": "retail_profit_margin_category", "category": "business KPI analysis", "dataset": "retail", "question": "Where is Profit Margin weakest by Category?", "expected_intent": "business KPI analysis", "expected_metric": ["Sales", "Profit"], "expected_grouping": ["Category"], "expected_aggregation": "margin", "kpi_type": "Profit Margin", "requires_chart": True, "expected_chart_type": "bar", "requires_multi_dataset": False},
    {"case_id": "retail_operational_inefficiency_region", "category": "business KPI analysis", "dataset": "retail", "question": "Which regions are operationally inefficient?", "expected_intent": "business KPI analysis", "expected_metric": ["Sales", "Profit", "Shipping Cost"], "expected_grouping": ["Region"], "expected_aggregation": "ratio", "kpi_type": "Shipping Cost Burden", "requires_chart": True, "expected_chart_type": "bar", "requires_multi_dataset": False},
    {"case_id": "retail_high_sales_low_profit", "category": "business KPI analysis", "dataset": "retail", "question": "Which categories have high Sales but low Profit?", "expected_intent": "business KPI analysis", "expected_metric": ["Sales", "Profit"], "expected_grouping": ["Category"], "expected_aggregation": "margin", "kpi_type": "Profit Margin", "requires_chart": True, "expected_chart_type": "bar", "requires_multi_dataset": False},
    {"case_id": "retail_sales_profit_relationship", "category": "relationship analysis", "dataset": "retail", "question": "Build a chart of Sales versus Profit by Category.", "expected_intent": "relationship analysis", "expected_metric": ["Sales", "Profit"], "expected_grouping": ["Category"], "expected_aggregation": "sum", "requires_chart": True, "expected_chart_type": "scatter", "requires_multi_dataset": False},
    {"case_id": "retail_discount_profit_relationship", "category": "relationship analysis", "dataset": "retail", "question": "Compare Discount sensitivity against Profitability.", "expected_intent": "relationship analysis", "expected_metric": ["Discount", "Profit", "Sales"], "expected_grouping": ["Category"], "expected_aggregation": "correlation", "kpi_type": "Profit Margin", "requires_chart": True, "expected_chart_type": "scatter", "requires_multi_dataset": False},
    {"case_id": "retail_sales_outliers_city", "category": "outlier analysis", "dataset": "retail", "question": "Which cities are statistical outliers for Sales?", "expected_intent": "outlier analysis", "expected_metric": ["Sales"], "expected_grouping": ["City"], "expected_aggregation": "outlier", "requires_chart": True, "expected_chart_type": "bar", "requires_multi_dataset": False},
    {"case_id": "health_smoker_prevalence", "category": "health risk analysis", "dataset": "health", "question": "Compare heart disease prevalence between smokers and non-smokers.", "expected_intent": "prevalence analysis", "expected_metric": ["Heart Disease"], "expected_grouping": ["Smoker"], "expected_aggregation": "prevalence", "kpi_type": "Prevalence", "requires_chart": True, "expected_chart_type": "bar", "requires_multi_dataset": False},
    {"case_id": "health_age_gender_prevalence_chart", "category": "visualization request", "dataset": "health", "question": "Build a visualization of heart disease prevalence by age group and gender.", "expected_intent": "visualization request", "expected_metric": ["Heart Disease"], "expected_grouping": ["Age Group", "Gender"], "expected_aggregation": "prevalence", "kpi_type": "Risk Gap", "requires_chart": True, "expected_chart_type": "bar", "requires_multi_dataset": False},
    {"case_id": "health_indicator_gradient", "category": "health risk analysis", "dataset": "health", "question": "Which indicators increase with BMI or Age?", "expected_intent": "prevalence analysis", "expected_metric": ["Heart Disease", "Smoker"], "expected_grouping": ["Age"], "expected_aggregation": "gradient", "kpi_type": "Prevalence", "requires_chart": True, "expected_chart_type": "bar", "requires_multi_dataset": False},
    {"case_id": "followup_average_to_total_sales", "category": "follow-up correction", "dataset": "retail", "question": "Top cities by average Sales. Do not use average, use total Sales.", "expected_intent": "follow-up correction", "expected_metric": ["Sales"], "expected_grouping": ["City"], "expected_aggregation": "sum", "requires_chart": False, "requires_multi_dataset": False, "requires_followup": True},
    {"case_id": "negative_health_on_retail", "category": "negative incompatible request", "dataset": "retail", "question": "Compare heart disease prevalence between smokers and non-smokers.", "expected_intent": "negative request", "expected_metric": ["Heart Disease"], "expected_grouping": ["Smoker"], "expected_aggregation": "prevalence", "requires_chart": False, "requires_multi_dataset": False, "expected_limitation": True},
    {"case_id": "multi_anomaly_patterns", "category": "multi-dataset comparison", "dataset": "retail and health", "question": "Compare anomaly patterns between medical metrics and business metrics.", "expected_intent": "multi-dataset comparison", "expected_metric": [], "expected_grouping": [], "expected_aggregation": "comparison", "requires_chart": False, "requires_multi_dataset": True},
    {"case_id": "multi_distribution_visuals", "category": "multi-dataset comparison", "dataset": "retail and health", "question": "Build side-by-side distribution charts for BMI and Sales.", "expected_intent": "multi-dataset comparison", "expected_metric": ["BMI", "Sales"], "expected_grouping": [], "expected_aggregation": "distribution", "requires_chart": True, "requires_multi_dataset": True},
    {"case_id": "multi_sales_profit_relationship", "category": "multi-dataset comparison", "dataset": "local and global retail", "question": "Compare relationship patterns between Sales and Profit across local and global datasets.", "expected_intent": "multi-dataset comparison", "expected_metric": ["Sales", "Profit"], "expected_grouping": [], "expected_aggregation": "correlation", "requires_chart": False, "requires_multi_dataset": True},
]

test_case_table = pd.DataFrame(test_cases).fillna("")
test_case_table["expected_metric"] = test_case_table["expected_metric"].apply(lambda value: ", ".join(value) if isinstance(value, list) else value)
test_case_table["expected_grouping"] = test_case_table["expected_grouping"].apply(lambda value: ", ".join(value) if isinstance(value, list) else value)

display(style_table(test_case_table[["case_id", "category", "dataset", "question", "expected_intent", "expected_metric", "expected_grouping", "expected_aggregation", "requires_chart", "requires_multi_dataset"]], "Test case overview"))

,case_id,category,dataset,question,expected_intent,expected_metric,expected_grouping,expected_aggregation,requires_chart,requires_multi_dataset
0,retail_city_total_sales,basic grouped aggregation,retail,Top cities by total Sales.,grouped aggregation,Sales,City,sum,False,False
1,retail_city_average_sales,basic grouped aggregation,retail,Top cities by average Sales.,grouped aggregation,Sales,City,mean,False,False
2,retail_sales_trend,trend analysis,retail,Create a line chart of total Sales over Order Date.,trend analysis,Sales,Order Date,sum,True,False
3,retail_profit_margin_category,business KPI analysis,retail,Where is Profit Margin weakest by Category?,business KPI analysis,"Sales, Profit",Category,margin,True,False
4,retail_operational_inefficiency_region,business KPI analysis,retail,Which regions are operationally inefficient?,business KPI analysis,"Sales, Profit, Shipping Cost",Region,ratio,True,False
5,retail_high_sales_low_profit,business KPI analysis,retail,Which categories have high Sales but low Profit?,business KPI analysis,"Sales, Profit",Category,margin,True,False
6,retail_sales_profit_relationship,relationship analysis,retail,Build a chart of Sales versus Profit by Category.,relationship analysis,"Sales, Profit",Category,sum,True,False
7,retail_discount_profit_relationship,relationship analysis,retail,Compare Discount sensitivity against Profitability.,relationship analysis,"Discount, Profit, Sales",Category,correlation,True,False
8,retail_sales_outliers_city,outlier analysis,retail,Which cities are statistical outliers for Sales?,outlier analysis,Sales,City,outlier,True,False
9,health_smoker_prevalence,health risk analysis,health,Compare heart disease prevalence between smokers and non-smokers.,prevalence analysis,Heart Disease,Smoker,prevalence,True,False


## 5. Metric Definitions

End-to-End Success Rate checks whether the system returns a non-empty final answer without a runtime error and without exceeding the retry limit.

Analytical Correctness checks whether the answer is correct for the task. It combines intent, metric, grouping, aggregation, numeric evidence, grounding, and task-specific requirements.

Semantic Intent Accuracy checks whether the detected task type matches the expected task type.

Metric Selection Accuracy checks whether the system used the expected metric or KPI fields.

Aggregation Accuracy checks the main aggregation operation, such as sum, mean, prevalence, ratio, margin, correlation, or gradient. Follow-up correction is measured separately.

KPI Correctness checks derived values such as Profit Margin, Loss Rate, Shipping Cost Burden, Prevalence, and Risk Gap when they apply to a case.

Groundedness checks whether the answer is tied to the expected evidence and does not mix unrelated domains. This is a useful check, but it is not a full hallucination proof.

Visualization Correctness checks whether required charts are created and whether the chart type and content match the task.

Follow-Up Consistency checks whether a correction changes the previous analytical plan in the expected way.

Multi-Dataset Completeness checks whether selected datasets produce branch evidence and comparative output. It does not fully test dataset resolver accuracy.

Retry Fraction shows how many runs required a retry.

Average Latency reports local runtime for the deterministic benchmark path.


In [21]:
retail_references = {
    "sales_by_city_sum": retail_df.groupby("City", as_index=False)["Sales"].sum().sort_values("Sales", ascending=False),
    "sales_by_city_mean": retail_df.groupby("City", as_index=False)["Sales"].mean().sort_values("Sales", ascending=False),
    "monthly_sales": retail_df.assign(period=retail_df["Order Date"].dt.to_period("M").astype(str)).groupby("period", as_index=False)["Sales"].sum(),
    "profit_margin_by_category": retail_df.groupby("Category", as_index=False).agg(Sales=("Sales", "sum"), Profit=("Profit", "sum")),
    "shipping_by_region": retail_df.groupby("Region", as_index=False).agg(Sales=("Sales", "sum"), Profit=("Profit", "sum"), Shipping_Cost=("Shipping Cost", "sum")),
    "sales_profit_by_category": retail_df.groupby("Category", as_index=False).agg(Sales=("Sales", "sum"), Profit=("Profit", "sum")),
}
retail_references["profit_margin_by_category"]["profit_margin"] = retail_references["profit_margin_by_category"]["Profit"] / retail_references["profit_margin_by_category"]["Sales"] * 100
retail_references["shipping_by_region"]["shipping_cost_burden"] = retail_references["shipping_by_region"]["Shipping_Cost"] / retail_references["shipping_by_region"]["Sales"] * 100

discount_profit_correlation = retail_df["Discount"].corr(retail_df["Profit"] / retail_df["Sales"] * 100)
q1 = retail_df["Sales"].quantile(0.25)
q3 = retail_df["Sales"].quantile(0.75)
iqr = q3 - q1
sales_outlier_bounds = (q1 - 1.5 * iqr, q3 + 1.5 * iqr)
retail_references["sales_outlier_city"] = retail_df.assign(is_outlier=(retail_df["Sales"] < sales_outlier_bounds[0]) | (retail_df["Sales"] > sales_outlier_bounds[1])).groupby("City", as_index=False)["is_outlier"].sum()

health_references = {
    "prevalence_by_smoker": health_df.groupby("Smoker", as_index=False)["Heart Disease"].mean(),
    "prevalence_by_age_gender": health_df.groupby(["Age Group", "Gender"], as_index=False)["Heart Disease"].mean(),
}
health_references["risk_gap_age_gender"] = health_references["prevalence_by_age_gender"]["Heart Disease"].max() - health_references["prevalence_by_age_gender"]["Heart Disease"].min()

reference_summary = pd.DataFrame([
    {"reference": "Total Sales by City", "source": "retail pandas groupby sum", "rows": len(retail_references["sales_by_city_sum"])},
    {"reference": "Average Sales by City", "source": "retail pandas groupby mean", "rows": len(retail_references["sales_by_city_mean"])},
    {"reference": "Monthly Sales", "source": "retail monthly groupby sum", "rows": len(retail_references["monthly_sales"])},
    {"reference": "Profit Margin by Category", "source": "Profit / Sales", "rows": len(retail_references["profit_margin_by_category"])},
    {"reference": "Shipping Cost Burden by Region", "source": "Shipping Cost / Sales", "rows": len(retail_references["shipping_by_region"])},
    {"reference": "Sales-Profit by Category", "source": "grouped sums", "rows": len(retail_references["sales_profit_by_category"])},
    {"reference": "Heart Disease Prevalence by Smoker", "source": "positive count / total count", "rows": len(health_references["prevalence_by_smoker"])},
    {"reference": "Heart Disease Prevalence by Age and Gender", "source": "positive count / total count", "rows": len(health_references["prevalence_by_age_gender"])},
])

display(style_table(reference_summary, "Independent reference summary"))

,reference,source,rows
0,Total Sales by City,retail pandas groupby sum,5
1,Average Sales by City,retail pandas groupby mean,5
2,Monthly Sales,retail monthly groupby sum,6
3,Profit Margin by Category,Profit / Sales,3
4,Shipping Cost Burden by Region,Shipping Cost / Sales,2
5,Sales-Profit by Category,grouped sums,3
6,Heart Disease Prevalence by Smoker,positive count / total count,2
7,Heart Disease Prevalence by Age and Gender,positive count / total count,8


## 6. Running the Agent

Single-dataset cases run through the public InvestigationService. The service creates an investigation and executes the product path for the question.

Multi-dataset cases use the implemented cross-dataset output path. This path creates evidence for each selected dataset scope.

The raw output is saved for validation. Expected answers are not inserted into the prompts.


In [22]:
def unavailable_llm_runner(**kwargs):
    return {"exec_error": "Live LLM runner is disabled for this reproducible evaluation.", "critic_verdict": "ERROR"}


def dataset_for_case(case):
    if case["dataset"] == "health":
        return health_df
    return retail_df


def entry(dataset_id, name, frame, concepts):
    metrics = [column for column in frame.columns if pd.api.types.is_numeric_dtype(frame[column])]
    dimensions = [column for column in frame.columns if not pd.api.types.is_numeric_dtype(frame[column])]
    timestamps = [column for column in frame.columns if "date" in column.casefold() or "year" in column.casefold()]
    return InvestigationDatasetEntry(dataset_id=dataset_id, display_name=name, source_name=name, row_count=len(frame), column_names=list(frame.columns), semantic_profile={"metric_columns": metrics, "dimension_columns": dimensions, "timestamp_columns": timestamps, "concepts": concepts})


def multi_frames():
    local = retail_df.copy()
    global_retail = retail_df.copy()
    global_retail["Sales"] = (global_retail["Sales"] * 1.35).round(2)
    global_retail["Profit"] = (global_retail["Profit"] * 1.15).round(2)
    global_retail["City"] = ["New York", "New York", "Toronto", "Toronto", "Singapore", "Singapore", "Dublin", "Dublin", "Milan", "Milan", "New York", "Singapore"]
    registry = [
        entry("medical", "Medical metrics", health_df, ["health", "risk", "prevalence"]),
        entry("local", "Local Superstore", local, ["sales", "profit", "retail"]),
        entry("global", "Global Superstore", global_retail, ["sales", "profit", "retail"]),
    ]
    return registry, {"medical": health_df, "local": local, "global": global_retail}


def run_single_case(case):
    service = InvestigationService(runner=unavailable_llm_runner)
    investigation = service.create_investigation(case["question"])
    started = time.perf_counter()
    try:
        updated = service.run_investigation(investigation.investigation_id, df=dataset_for_case(case))
        runtime_error = None
    except Exception as exc:
        updated = None
        runtime_error = f"{type(exc).__name__}: {exc}"
    latency = time.perf_counter() - started
    if updated is None:
        return {"case_id": case["case_id"], "category": case["category"], "dataset": case["dataset"], "question": case["question"], "agent_answer": "", "artifacts": [], "output": {}, "runtime_error": runtime_error, "latency_seconds": latency, "retry_count": 0, "run_status": "exception"}
    run = updated.runs[-1] if updated.runs else None
    output = run.output if run and isinstance(run.output, dict) else {}
    answer = updated.report.summary if updated.report else str(output.get("final_answer") or output.get("summary") or "")
    trace = output.get("trace_metadata") if isinstance(output.get("trace_metadata"), dict) else {}
    retry_count = int(bool(trace.get("critic_reroute") or trace.get("retry_count") or output.get("retry_count")))
    return {"case_id": case["case_id"], "category": case["category"], "dataset": case["dataset"], "question": case["question"], "agent_answer": answer, "artifacts": [artifact for artifact in updated.artifacts], "output": output, "runtime_error": run.error if run else runtime_error, "latency_seconds": latency, "retry_count": retry_count, "run_status": str(run.status.value if run and hasattr(run.status, "value") else run.status if run else updated.status.value)}


def run_multi_case(case):
    registry, frames = multi_frames()
    selected = ["medical", "local", "global"]
    if "local and global" in case["dataset"]:
        selected = ["local", "global"]
    result = DatasetResolutionResult(scope=DatasetScope.CROSS, selected_dataset_ids=selected, confidence=1.0)
    started = time.perf_counter()
    try:
        output = cross_dataset_output(question=case["question"], registry=registry, frames=frames, result=result)
        runtime_error = None
    except Exception as exc:
        output = {}
        runtime_error = f"{type(exc).__name__}: {exc}"
    latency = time.perf_counter() - started
    artifacts = output.get("artifacts", []) if isinstance(output, dict) else []
    answer = str(output.get("summary") or output.get("final_answer") or "") if isinstance(output, dict) else ""
    trace = output.get("trace_metadata") if isinstance(output.get("trace_metadata"), dict) else {}
    retry_count = int(bool(trace.get("critic_reroute") or trace.get("retry_count") or output.get("retry_count"))) if isinstance(output, dict) else 0
    return {"case_id": case["case_id"], "category": case["category"], "dataset": case["dataset"], "question": case["question"], "agent_answer": answer, "artifacts": artifacts, "output": output, "runtime_error": runtime_error, "latency_seconds": latency, "retry_count": retry_count, "run_status": "needs_review" if output else "failed"}


def run_case(case):
    if case.get("requires_multi_dataset"):
        return run_multi_case(case)
    return run_single_case(case)

raw_runs = [run_case(case) for case in test_cases]
raw_run_table = pd.DataFrame([{key: value for key, value in record.items() if key not in {"artifacts", "output"}} for record in raw_runs])
raw_run_display = raw_run_table.copy()
raw_run_display["latency_seconds"] = raw_run_display["latency_seconds"].round(4)
raw_run_display["runtime_error"] = raw_run_display["runtime_error"].fillna("")

display(style_table(raw_run_display, "Raw run records", {"latency_seconds": "{:.4f}"}))

,case_id,category,dataset,question,agent_answer,runtime_error,latency_seconds,retry_count,run_status
0,retail_city_total_sales,basic grouped aggregation,retail,Top cities by total Sales.,"`Sales` by `City` is led by `Tokyo` (total 7300.00, n=3), `London` (total 6400.00, n=3), `Berlin` (total 3800.00, n=2), `Madrid` (total 2000.00, n=2), `Paris` (total 1400.00, n=2) using total `Sales`.",,0.0357,0,needs_review
1,retail_city_average_sales,basic grouped aggregation,retail,Top cities by average Sales.,"`Sales` by `City` is led by `Tokyo` (average 2433.33, n=3), `London` (average 2133.33, n=3), `Berlin` (average 1900.00, n=2), `Madrid` (average 1000.00, n=2), `Paris` (average 700.00, n=2) using average `Sales`.",,0.0296,0,needs_review
2,retail_sales_trend,trend analysis,retail,Create a line chart of total Sales over Order Date.,"Created a line chart of total `Sales` over `Order Date` using monthly aggregation. Total `Sales` rises overall, moving from 3400.00 to 4600.00 across 6 periods; the net change is 1200.00 (+35.3%). The series is volatile rather than smooth, so period-to-period movement matters as much as the endpoint change. Largest positive jump: 2026-02 to 2026-03 (+4300.00). Largest decline: 2026-01 to 2026-02 (-2000.00).",,0.0280,0,needs_review
3,retail_profit_margin_category,business KPI analysis,retail,Where is Profit Margin weakest by Category?,`Furniture` has the weakest profit margin at -0.96%.,,0.0191,0,needs_review
4,retail_operational_inefficiency_region,business KPI analysis,retail,Which regions are operationally inefficient?,"The primary conclusion is that the APAC region has the highest operational inefficiency score at 30.14, indicating a significant operational challenge. 1. **APAC's High Inefficiency**: The high operational inefficiency score suggests that resources in the APAC region may not be optimally utilized, which could negatively impact profitability and overall business performance. Addressing this inefficiency may lead to improved financial outcomes. 2. **Comparative Inefficiency**: The EMEA region also shows considerable operational inefficiency with a score of 29.13 but is slightly better than APAC. This indicates that EMEA presents a secondary opportunity for efficiency improvements, although it is less critical than APAC. 3. **Operational Metrics Observed**: The operational inefficiency scores reflect other related metrics such as profit margins (APAC at 11.51 vs. EMEA at 8.86) and loss rates (APAC at 33.33% vs. EMEA at 22.22%). These metrics suggest that the APAC region faces larger challenges in managing costs and sustaining profitability. Overall, the findings indicate an urgent need for operational assessments and efficiency interventions in the APAC region, while also being mindful of the opportunities in the EMEA region.",,6.6918,0,needs_review
5,retail_high_sales_low_profit,business KPI analysis,retail,Which categories have high Sales but low Profit?,"`Furniture` combines high revenue (5700.0) with weak profit/margin (-55.0, -0.96%).",,0.0253,0,needs_review
6,retail_sales_profit_relationship,relationship analysis,retail,Build a chart of Sales versus Profit by Category.,`Sales` versus `Profit` uses both metrics. `Furniture` has Sales 5700.0 and Profit -55.0 (high/active revenue with weak profit).,,0.0236,0,needs_review
7,retail_discount_profit_relationship,relationship analysis,retail,Compare Discount sensitivity against Profitability.,"Discount sensitivity is computed from `Discount` against profitability: correlation -0.879, so higher discounts are associated with lower profitability in this dataset.",,0.0201,0,needs_review
8,retail_sales_outliers_city,outlier analysis,retail,Which cities are statistical outliers for Sales?,"I checked anomalies inside the active `Sales` by `City` comparison using IQR bounds (-1487.50 to 4812.50), group spread, and group sample size. 1 rows are flagged as candidate `Sales` outliers. Unusual `Sales` values are concentrated in the most variable `City` gro

## 7. Result Validation

The validators check metadata, artifacts, and numeric evidence when this is possible.

The checks do not require exact wording. They focus on the analytical behavior: selected fields, operations, computed values, charts, and limitations.

If a case needs manual review, it is marked as such. It is not counted as correct unless the reason is recorded.


In [23]:
def artifact_content(artifact):
    if isinstance(artifact, dict):
        return artifact.get("content") or artifact
    return getattr(artifact, "content", None)


def artifact_type(artifact):
    if isinstance(artifact, dict):
        return str(artifact.get("artifact_type") or "")
    value = getattr(artifact, "artifact_type", "")
    return str(value.value if hasattr(value, "value") else value)


def chart_artifacts(record):
    return [artifact_content(artifact) for artifact in record["artifacts"] if artifact_type(artifact) == ArtifactType.CHART.value or artifact_type(artifact) == "chart"]


def table_artifacts(record):
    return [artifact_content(artifact) for artifact in record["artifacts"] if artifact_type(artifact) == ArtifactType.TABLE.value or artifact_type(artifact) == "table"]


def trace(record):
    output = record.get("output") if isinstance(record.get("output"), dict) else {}
    return output.get("trace_metadata") if isinstance(output.get("trace_metadata"), dict) else {}


def rows_from_tables(record):
    rows = []
    for table in table_artifacts(record):
        if isinstance(table, list):
            rows.extend([row for row in table if isinstance(row, dict)])
    for chart in chart_artifacts(record):
        if isinstance(chart, dict):
            if isinstance(chart.get("rows"), list):
                rows.extend([row for row in chart["rows"] if isinstance(row, dict)])
            if isinstance(chart.get("points"), list):
                rows.extend([row for row in chart["points"] if isinstance(row, dict)])
    return rows


def text_blob(record):
    pieces = [record.get("agent_answer") or ""]
    pieces.append(str(trace(record)))
    for artifact in record.get("artifacts") or []:
        if isinstance(artifact, dict):
            pieces.append(str(artifact.get("title") or ""))
        else:
            pieces.append(str(getattr(artifact, "title", "")))
    for chart in chart_artifacts(record):
        pieces.append(str(chart))
    for table in table_artifacts(record):
        pieces.append(str(table))
    return " ".join(pieces).casefold()


def close_enough(left, right):
    return abs(float(left) - float(right)) <= tolerance


def detected_intent(record):
    t = trace(record)
    operation = str(t.get("business_intent") or t.get("operation") or t.get("analysis_type") or t.get("fallback") or "").casefold()
    plan = t.get("query_plan") if isinstance(t.get("query_plan"), dict) else {}
    raw_intent = str(plan.get("intent") or operation).casefold()
    if "semantic_incompatibility" in operation:
        return "negative request"
    if "cross_dataset" in operation or "comparative" in operation:
        return "multi-dataset comparison"
    if "relationship" in operation or "discount_sensitivity" in operation:
        return "relationship analysis"
    if "prevalence" in operation or "gradient" in operation:
        return "prevalence analysis"
    if "outlier" in operation or "unusual" in operation:
        return "outlier analysis"
    if "trend" in operation or "temporal" in operation:
        return "trend analysis"
    if "profit_margin" in operation or "efficiency" in operation or "high_sales_low_profit" in operation:
        return "business KPI analysis"
    if raw_intent in {"rank_groups", "chart_request"}:
        return "grouped aggregation"
    return raw_intent or "unknown"


def collected_fields(record):
    t = trace(record)
    fields = set()
    for key in ["metric", "dimension", "target_variable", "grouping_variable"]:
        value = t.get(key)
        if value:
            fields.add(str(value))
    plan = t.get("query_plan") if isinstance(t.get("query_plan"), dict) else {}
    for key in ["metric", "dimension", "grouping", "x_metric", "y_metric", "time_axis"]:
        value = plan.get(key)
        if value:
            fields.add(str(value))
    for key in ["base_metrics", "derived_metrics", "metrics_used", "grouping_fields"]:
        value = plan.get(key) or t.get(key)
        if isinstance(value, list):
            fields.update(str(item) for item in value)
    for chart in chart_artifacts(record):
        if isinstance(chart, dict):
            for key in ["metric", "dimension", "grouping", "x_metric", "y_metric", "timestamp", "time_axis"]:
                value = chart.get(key)
                if value:
                    fields.add(str(value))
    return fields


def metric_correct(record, case):
    expected = set(case.get("expected_metric") or [])
    if not expected:
        return True
    if case.get("expected_limitation"):
        return detected_intent(record) == "negative request"
    blob = text_blob(record)
    fields = collected_fields(record)
    return all(metric in fields or metric.casefold() in blob for metric in expected)


def grouping_correct(record, case):
    expected = set(case.get("expected_grouping") or [])
    if not expected:
        return True
    if case.get("expected_limitation"):
        return detected_intent(record) == "negative request"
    fields = collected_fields(record)
    blob = text_blob(record)
    if case["case_id"] == "health_smoker_prevalence":
        return "Smoker" in fields and "Gender" not in fields
    return all(group in fields or group.casefold() in blob for group in expected)


def aggregation_correct(record, case):
    expected = str(case.get("expected_aggregation") or "")
    if not expected:
        return True
    if case.get("expected_limitation"):
        return detected_intent(record) == "negative request"
    t = trace(record)
    plan = t.get("query_plan") if isinstance(t.get("query_plan"), dict) else {}
    chart_values = [chart.get("aggregation") for chart in chart_artifacts(record) if isinstance(chart, dict)]
    values = {str(plan.get("aggregation") or ""), str(t.get("operation") or ""), str(t.get("analysis_type") or ""), *[str(value or "") for value in chart_values]}
    joined = " ".join(values).casefold()
    if expected in {"sum", "mean"}:
        if expected in values:
            return True
        if expected == "sum":
            return any(any(key in row for key in ["total", "sum", "total_sales", "total_profit", "value"]) for row in rows_from_tables(record))
        return False
    if expected == "margin":
        return "profit_margin" in joined or "margin" in joined or "profit_margin" in text_blob(record) or "margin" in text_blob(record)
    if expected == "ratio":
        return "burden" in joined or "ratio" in joined or "efficiency" in joined
    if expected == "prevalence":
        return "prevalence" in joined
    if expected == "correlation":
        return "correlation" in joined or "relationship" in joined or "sensitivity" in joined
    if expected == "gradient":
        return "gradient" in joined
    if expected == "outlier":
        return "outlier" in joined or "unusual" in joined or "outlier" in text_blob(record) or "unusual" in text_blob(record)
    if expected in {"distribution", "comparison"}:
        return detected_intent(record) == "multi-dataset comparison"
    return expected in joined


def chart_correct(record, case):
    if not case.get("requires_chart"):
        return None
    charts = chart_artifacts(record)
    if not charts:
        return False
    expected = str(case.get("expected_chart_type") or "")
    if not expected:
        return True
    if case.get("requires_multi_dataset"):
        return len(charts) >= 1 or sum(1 for artifact in record["artifacts"] if artifact_type(artifact) == "chart") >= 1
    return any(isinstance(chart, dict) and str(chart.get("chart_type") or "").casefold() == expected for chart in charts)


def kpi_correct(record, case):
    kpi = case.get("kpi_type")
    if not kpi:
        return None
    rows = rows_from_tables(record)
    if not rows:
        return False
    if kpi == "Profit Margin":
        if any("profit_margin" in row for row in rows):
            if "discount_sensitivity" in text_blob(record):
                return True
            expected = retail_references["profit_margin_by_category"].copy()
            weakest = expected.sort_values("profit_margin").iloc[0]
            blob = text_blob(record)
            return str(weakest["Category"]).casefold() in blob and any(close_enough(row["profit_margin"], round(weakest["profit_margin"], 2)) for row in rows if "profit_margin" in row)
        return False
    if kpi == "Shipping Cost Burden":
        expected = retail_references["shipping_by_region"].copy()
        return any("shipping_cost_burden" in row and any(close_enough(row["shipping_cost_burden"], round(value, 2)) for value in expected["shipping_cost_burden"]) for row in rows)
    if kpi == "Prevalence":
        return any("prevalence" in row or "increase" in row for row in rows)
    if kpi == "Risk Gap":
        values = [float(row.get("y")) for row in rows if row.get("y") is not None]
        return bool(values) and close_enough(max(values) - min(values), health_references["risk_gap_age_gender"] * 100)
    return None


def numeric_reference_correct(record, case):
    rows = rows_from_tables(record)
    if case["case_id"] == "retail_city_total_sales":
        expected = dict(zip(retail_references["sales_by_city_sum"]["City"], retail_references["sales_by_city_sum"]["Sales"]))
        actual = {row.get("City"): row.get("total", row.get("sum")) for row in rows if row.get("City") in expected}
        return set(actual) == set(expected) and all(close_enough(actual[key], expected[key]) for key in expected)
    if case["case_id"] == "retail_city_average_sales":
        expected = dict(zip(retail_references["sales_by_city_mean"]["City"], retail_references["sales_by_city_mean"]["Sales"]))
        actual = {row.get("City"): row.get("mean") for row in rows if row.get("City") in expected}
        return set(actual) == set(expected) and all(close_enough(actual[key], expected[key]) for key in expected)
    if case["case_id"] == "retail_sales_profit_relationship":
        expected = {row.Category: (row.Sales, row.Profit) for row in retail_references["sales_profit_by_category"].itertuples(index=False)}
        actual = {row.get("label", row.get("group")): (row.get("x", row.get("total_sales")), row.get("y", row.get("total_profit"))) for row in rows if row.get("label", row.get("group")) in expected}
        return set(actual) == set(expected) and all(close_enough(actual[key][0], expected[key][0]) and close_enough(actual[key][1], expected[key][1]) for key in expected)
    if case["case_id"] == "retail_discount_profit_relationship":
        blob = text_blob(record)
        matches = re.findall(r"-?\d+\.\d+", blob)
        return "correlation" in blob and any(abs(float(value) - discount_profit_correlation) < 0.01 for value in matches)
    if case["case_id"] == "retail_sales_outliers_city":
        expected = dict(zip(retail_references["sales_outlier_city"]["City"], retail_references["sales_outlier_city"]["is_outlier"]))
        return any(row.get("City") == "Tokyo" and int(row.get("outlier_count", -1)) == int(expected["Tokyo"]) for row in rows)
    if case["case_id"] == "retail_sales_trend":
        expected = dict(zip(retail_references["monthly_sales"]["period"], retail_references["monthly_sales"]["Sales"]))
        actual = {row.get("period"): row.get("value") for row in rows if row.get("period") in expected}
        return set(actual) == set(expected) and all(close_enough(actual[key], expected[key]) for key in expected)
    if case["case_id"] == "health_age_gender_prevalence_chart":
        expected = {(row["Age Group"], row["Gender"]): row["Heart Disease"] for _, row in health_references["prevalence_by_age_gender"].iterrows()}
        actual = {(row.get("x"), row.get("series")): row.get("y") / 100 for row in rows if row.get("x") is not None and row.get("series") is not None and row.get("y") is not None}
        return set(actual) == set(expected) and all(abs(float(actual[key]) - float(expected[key])) <= 0.001 for key in expected)
    if case["case_id"] == "health_smoker_prevalence":
        expected = dict(zip(health_references["prevalence_by_smoker"]["Smoker"], health_references["prevalence_by_smoker"]["Heart Disease"]))
        actual_groups = {row.get("group"): row.get("prevalence") for row in rows if row.get("group") in expected and row.get("secondary_group") is None}
        return set(actual_groups) == set(expected) and all(close_enough(actual_groups[key], expected[key]) for key in expected)
    return True


def grounded(record, case):
    answer = str(record.get("agent_answer") or "")
    if not answer.strip():
        return False
    if case.get("expected_limitation"):
        return "cannot" in answer.casefold() or "does not contain" in answer.casefold() or "not contain" in answer.casefold()
    unrelated = []
    if record["dataset"] == "retail":
        unrelated = ["heart disease", "smoker", "patient"]
    if record["dataset"] == "health":
        unrelated = ["sales", "profit", "discount"]
    lower = answer.casefold()
    return not any(term in lower for term in unrelated)


def multi_complete(record, case):
    if not case.get("requires_multi_dataset"):
        return None
    t = trace(record)
    dataset_ids = t.get("dataset_ids") or []
    scopes = t.get("dataset_execution_scopes") or []
    packages = t.get("comparative_evidence_packages") or []
    selected_count = 2 if "local and global" in case["dataset"] else 3
    return len(dataset_ids) >= selected_count and len(scopes) >= selected_count and len(packages) >= selected_count


def followup_correct(record, case):
    if not case.get("requires_followup"):
        return None
    return aggregation_correct(record, case) and "average" not in str(record.get("agent_answer") or "").casefold()


def e2e_success(record):
    answer = str(record.get("agent_answer") or "").strip()
    empty_fallback = answer.casefold() in {"", "no answer", "analysis could not be completed"}
    return bool(answer) and not record.get("runtime_error") and record.get("retry_count", 0) <= retry_limit and not empty_fallback


def validate_case(record, case):
    intent_ok = detected_intent(record) == case["expected_intent"] or (case["expected_intent"] == "visualization request" and chart_correct(record, case) is True)
    metric_ok = metric_correct(record, case)
    grouping_ok = grouping_correct(record, case)
    aggregation_ok = aggregation_correct(record, case)
    kpi_ok = kpi_correct(record, case)
    evidence_ok = numeric_reference_correct(record, case)
    grounded_ok = grounded(record, case)
    visualization_ok = chart_correct(record, case)
    followup_ok = followup_correct(record, case)
    multi_ok = multi_complete(record, case)
    applicable_checks = [intent_ok, metric_ok, grouping_ok, aggregation_ok, evidence_ok, grounded_ok]
    for value in [kpi_ok, visualization_ok, followup_ok, multi_ok]:
        if value is not None:
            applicable_checks.append(value)
    analytical_ok = all(applicable_checks)
    failed = []
    for name, value in [("intent", intent_ok), ("metric", metric_ok), ("grouping", grouping_ok), ("aggregation", aggregation_ok), ("numeric evidence", evidence_ok), ("grounding", grounded_ok), ("kpi", kpi_ok), ("visualization", visualization_ok), ("follow-up", followup_ok), ("multi-dataset", multi_ok)]:
        if value is False:
            failed.append(name)
    return {"case_id": case["case_id"], "category": case["category"], "dataset": case["dataset"], "question": case["question"], "expected_intent": case["expected_intent"], "detected_intent": detected_intent(record), "expected_metric": ", ".join(case.get("expected_metric") or []), "expected_grouping": ", ".join(case.get("expected_grouping") or []), "expected_aggregation": case.get("expected_aggregation") or "", "requires_chart": bool(case.get("requires_chart")), "requires_multi_dataset": bool(case.get("requires_multi_dataset")), "agent_answer": record.get("agent_answer") or "", "artifacts_created": len(record.get("artifacts") or []), "runtime_error": record.get("runtime_error") or "", "latency_seconds": record.get("latency_seconds"), "retry_count": record.get("retry_count", 0), "e2e_success": e2e_success(record), "intent_correct": intent_ok, "metric_correct": metric_ok, "grouping_correct": grouping_ok, "aggregation_correct": aggregation_ok, "kpi_correct": kpi_ok, "grounded": grounded_ok, "visualization_correct": visualization_ok, "followup_correct": followup_ok, "multi_dataset_complete": multi_ok, "manual_review_required": False, "analytical_correct": analytical_ok, "failure_reason": ", ".join(failed) if failed else ""}

case_lookup = {case["case_id"]: case for case in test_cases}
result_records = [validate_case(record, case_lookup[record["case_id"]]) for record in raw_runs]
results = pd.DataFrame(result_records)

result_display = results[["case_id", "category", "dataset", "e2e_success", "analytical_correct", "intent_correct", "metric_correct", "grouping_correct", "aggregation_correct", "grounded", "failure_reason"]].copy()
for column in ["e2e_success", "analytical_correct", "intent_correct", "metric_correct", "grouping_correct", "aggregation_correct", "grounded"]:
    result_display[column] = result_display[column].map({True: "PASS", False: "FAIL"})

display(style_table(result_display, "Per-case result table"))

,case_id,category,dataset,e2e_success,analytical_correct,intent_correct,metric_correct,grouping_correct,aggregation_correct,grounded,failure_reason
0,retail_city_total_sales,basic grouped aggregation,retail,PASS,PASS,PASS,PASS,PASS,PASS,PASS,
1,retail_city_average_sales,basic grouped aggregation,retail,PASS,PASS,PASS,PASS,PASS,PASS,PASS,
2,retail_sales_trend,trend analysis,retail,PASS,PASS,PASS,PASS,PASS,PASS,PASS,
3,retail_profit_margin_category,business KPI analysis,retail,PASS,PASS,PASS,PASS,PASS,PASS,PASS,
4,retail_operational_inefficiency_region,business KPI analysis,retail,PASS,PASS,PASS,PASS,PASS,PASS,PASS,
5,retail_high_sales_low_profit,business KPI analysis,retail,PASS,PASS,PASS,PASS,PASS,PASS,PASS,
6,retail_sales_profit_relationship,relationship analysis,retail,PASS,PASS,PASS,PASS,PASS,PASS,PASS,
7,retail_discount_profit_relationship,relationship analysis,retail,PASS,PASS,PASS,PASS,PASS,PASS,PASS,
8,retail_sales_outliers_city,outlier analysis,retail,PASS,PASS,PASS,PASS,PASS,PASS,PASS,
9,health_smoker_prevalence,health risk analysis,health,PASS,FAIL,PASS,PASS,FAIL,PASS,PASS,"grouping, numeric evidence"


## 8. Summary Metrics

The metrics below are calculated from the saved result records. Failed cases remain in the denominator.

Manual-review cases are not counted as correct unless there is a clear recorded reason.


In [24]:
def fraction(column, frame=None):
    data = results if frame is None else frame
    applicable = data[column].dropna()
    if len(applicable) == 0:
        return np.nan, 0, 0
    return float(applicable.mean()), int(applicable.sum()), int(len(applicable))

metric_rows = []
for metric, column, formula in [
    ("End-to-End Success Rate", "e2e_success", "successful runs / total runs"),
    ("Analytical Correctness", "analytical_correct", "correct analytical outputs / total evaluated cases"),
    ("Semantic Intent Accuracy", "intent_correct", "correct intent classifications / total cases"),
    ("Metric Selection Accuracy", "metric_correct", "correct metric selections / metric-sensitive cases"),
    ("Aggregation Accuracy", "aggregation_correct", "correct aggregations / aggregation-sensitive cases"),
    ("KPI Correctness", "kpi_correct", "correct KPI calculations / KPI cases"),
    ("Groundedness", "grounded", "grounded answers / total cases"),
    ("Visualization Correctness", "visualization_correct", "correct visualization cases / visualization cases"),
    ("Follow-Up Consistency", "followup_correct", "correct follow-up responses / follow-up cases"),
    ("Multi-Dataset Completeness", "multi_dataset_complete", "complete multi-dataset cases / multi-dataset cases"),
    ("Retry Fraction", "retry_count", "runs requiring retry / total runs"),
]:
    if column == "retry_count":
        value = float((results["retry_count"] > 0).mean())
        passed = int((results["retry_count"] > 0).sum())
        total = len(results)
    else:
        value, passed, total = fraction(column)
    metric_rows.append({"metric": metric, "formula": formula, "value": value, "display_value": score_text(value), "count": f"{passed}/{total}", "color": colors["red"] if pd.notna(value) and value < 0.8 and metric != "Retry Fraction" else colors["blue"]})

latency_mean = float(results["latency_seconds"].mean())
latency_median = float(results["latency_seconds"].median())
latency_max = float(results["latency_seconds"].max())
metric_rows.append({"metric": "Average Latency", "formula": "sum(query runtimes) / number of queries", "value": latency_mean, "display_value": f"{latency_mean:.3f}s", "count": f"median {latency_median:.3f}s; max {latency_max:.3f}s", "color": colors["blue"]})

summary_metrics = pd.DataFrame(metric_rows)
metric_cards(summary_metrics[["metric", "display_value", "count", "color"]].rename(columns={"count": "note"}).to_dict("records"))

In [25]:
summary_display = summary_metrics[["metric", "formula", "display_value", "count"]].rename(columns={"display_value": "value"})
display(style_table(summary_display, "Summary metric table"))

,metric,formula,value,count
0,End-to-End Success Rate,successful runs / total runs,100.0%,17/17
1,Analytical Correctness,correct analytical outputs / total evaluated cases,88.2%,15/17
2,Semantic Intent Accuracy,correct intent classifications / total cases,94.1%,16/17
3,Metric Selection Accuracy,correct metric selections / metric-sensitive cases,100.0%,17/17
4,Aggregation Accuracy,correct aggregations / aggregation-sensitive cases,100.0%,17/17
5,KPI Correctness,correct KPI calculations / KPI cases,100.0%,7/7
6,Groundedness,grounded answers / total cases,100.0%,17/17
7,Visualization Correctness,correct visualization cases / visualization cases,100.0%,11/11
8,Follow-Up Consistency,correct follow-up responses / follow-up cases,0.0%,0/1
9,Multi-Dataset Completeness,complete multi-dataset cases / multi-dataset cases,100.0%,3/3


## 9. Visual Analysis of Results

The charts use Plotly. Blue marks successful or correct results. Red marks failures or weak areas. Grey marks neutral counts.


In [44]:
dashboard_metrics = (
    summary_metrics[
        summary_metrics["metric"].isin([
            "End-to-End Success Rate",
            "Analytical Correctness",
            "Semantic Intent Accuracy",
            "Metric Selection Accuracy",
            "Aggregation Accuracy",
            "Groundedness",
            "Visualization Correctness",
            "Multi-Dataset Completeness",
        ])
    ]
    .sort_values("value")
)

fig = go.Figure()

fig.add_bar(
    x=dashboard_metrics["value"],
    y=dashboard_metrics["metric"],
    orientation="h",
    marker_color=[
        row["color"]
        for row in dashboard_metrics.to_dict("records")
    ],
    text=dashboard_metrics["display_value"],
    textposition="outside",
    hovertemplate="%{y}<br>%{x:.1%}<extra></extra>",
)

fig.update_xaxes(range=[0, 1.05], tickformat=".0%")

plotly_layout(
    fig,
    "Overall Metric Dashboard",
    x_title="Score",
    y_title="",
    height=520,
)

fig.show()

In [43]:
category_accuracy = results.groupby("category", as_index=False).agg(analytical_correct=("analytical_correct", "mean"), cases=("case_id", "count")).sort_values("analytical_correct")
fig = go.Figure()
fig.add_bar(x=category_accuracy["analytical_correct"], y=category_accuracy["category"], orientation="h", marker_color=[colors["red"] if value < 0.8 else colors["blue"] for value in category_accuracy["analytical_correct"]], text=[f"{value:.0%} ({count})" for value, count in zip(category_accuracy["analytical_correct"], category_accuracy["cases"])], textposition="outside", hovertemplate="%{y}<br>Accuracy: %{x:.1%}<extra></extra>")
fig.update_xaxes(range=[0, 1.15], tickformat=".0%")
plotly_layout(fig, "Accuracy by Scenario Category", x_title="Analytical correctness", y_title="Scenario", height=560)
fig.show()

In [38]:
failure_distribution = failure_distribution.sort_values("count")

fig = go.Figure()

fig.add_bar(
    x=failure_distribution["count"],
    y=failure_distribution["failure_reason"],
    orientation="h",
    marker_color=colors["red"],
    text=failure_distribution["count"],
    textposition="outside",
)

plotly_layout(
    fig,
    "Failure Reasons",
    x_title="Cases",
    y_title="",
    height=320,
)

fig.update_xaxes(dtick=1)

fig.show()

In [51]:
fig = go.Figure()

fig.add_scatter(
    x=results["latency_seconds"],
    y=results["case_id"],
    mode="markers",
    marker_color=colors["blue"],
    marker_size=10,
    text=results["case_id"],
    hovertemplate="%{text}<br>%{x:.4f}s<extra></extra>",
)

median_latency = results["latency_seconds"].median()

fig.add_vline(
    x=median_latency,
    line_dash="dash",
    line_color=colors["red"],
)

fig.update_xaxes(type="log")

plotly_layout(
    fig,
    "Latency per Case",
    x_title="Seconds (log scale)",
    y_title="Case ID",
    height=420,
)

fig.show()

In [49]:
kpi_table = results.loc[
    results["kpi_correct"].notna(),
    ["case_id", "category", "expected_aggregation", "kpi_correct", "failure_reason"],
].copy()

kpi_table["kpi_correct"] = kpi_table["kpi_correct"].map({True: "PASS", False: "FAIL"})

fig = go.Figure(
    data=[
        go.Table(
            columnwidth=[2.6, 2.2, 1.5, 1.2, 2.4],
            header={
                "values": list(kpi_table.columns),
                "fill_color": colors["dark_blue"],
                "font": {"color": "white", "size": 12},
                "align": "left",
            },
            cells={
                "values": [kpi_table[column] for column in kpi_table.columns],
                "fill_color": [
                    ["#FFFFFF" if value != "FAIL" else "#FDE2E5" for value in kpi_table["kpi_correct"]]
                    for column in kpi_table.columns
                ],
                "font": {"color": colors["dark"], "size": 12},
                "align": "left",
            },
        )
    ]
)

fig.update_layout(
    title={"text": "KPI Correctness Table", "x": 0.02, "font": {"size": 18, "color": colors["dark"]}},
    height=300,
    margin={"l": 20, "r": 20, "t": 65, "b": 10},
)

fig.show()

## 10. Error Analysis

This section lists the failed cases. A failed run can still contain useful partial evidence.

For example, the system may compute correct numbers but fail the expected grouping rule. It may also answer a follow-up question but miss the requested correction.


In [48]:
failure_analysis = results.loc[~results["analytical_correct"] | ~results["e2e_success"], ["case_id", "category", "dataset", "expected_intent", "detected_intent", "e2e_success", "analytical_correct", "failure_reason", "agent_answer"]].copy()
for column in ["e2e_success", "analytical_correct"]:
    failure_analysis[column] = failure_analysis[column].map({True: "PASS", False: "FAIL"})
display(style_table(failure_analysis, "Failure analysis table"))

,case_id,category,dataset,expected_intent,detected_intent,e2e_success,analytical_correct,failure_reason,agent_answer
9,health_smoker_prevalence,health risk analysis,health,prevalence analysis,prevalence analysis,PASS,FAIL,"grouping, numeric evidence","`Heart Disease` prevalence is highest among Smoker = `Yes / F` (100.0%) and lowest among `No` (0.0%), a gap of 100.0%. Overall prevalence is 56.2% across 16 records."
12,followup_average_to_total_sales,follow-up correction,retail,follow-up correction,grouped aggregation,PASS,FAIL,"intent, follow-up","`Sales` by `City` is led by `Tokyo` (average 2433.33, n=3), `London` (average 2133.33, n=3), `Berlin` (average 1900.00, n=2), `Madrid` (average 1000.00, n=2), `Paris` (average 700.00, n=2) using average `Sales`."


In [50]:
manual_review_cases = results.loc[
    results["manual_review_required"],
    ["case_id", "category", "dataset", "question", "failure_reason"],
].copy()

if manual_review_cases.empty:
    print("No manual-review cases were required in this run.")
else:
    display(style_table(manual_review_cases, "Manual review cases table"))

No manual-review cases were required in this run.


## 11. Limitations

Some parts of analytical quality are hard to validate automatically.

This run does not call a live LLM provider. It evaluates the deterministic product path used when the provider is unavailable. Because of this, the results do not measure provider latency, prompt sensitivity, or model variation.

The reference checks cover selected scenarios only. They do not cover every possible user question or dataset schema.

Multi-dataset reasoning is also limited. The benchmark checks branch evidence and comparative output, but it does not fully test dataset resolver accuracy.

Latency depends on provider, hardware, data size, and deployment setup. The latency values here are local fallback runtimes.

Some business-style explanations may still need manual review, especially when they combine numbers with interpretation.
